In [1]:
from __future__ import absolute_import
from __future__  import division
from __future__ import print_function
import tensorflow as tf
import numpy as np
from skimage.io import imread
from skimage.transform import resize
import cv2
import numpy as np
import os
from PIL import Image
from io import BytesIO
import time



In [2]:
from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())

[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 4125276198260006869
xla_global_id: -1
]


In [16]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers

def mamon_videoFightModel2():
    num_classes = 2
    
    # Define the VGG19 base model with pre-trained ImageNet weights
    base_model = tf.keras.applications.VGG19(include_top=False, weights='imagenet', input_shape=(160, 160, 3))
    
    # Freeze the layers of the base model
    for layer in base_model.layers:
        layer.trainable = False
    
    # Define the CNN part of the model
    cnn = models.Sequential()
    cnn.add(base_model)
    cnn.add(layers.Flatten())

    # Define the full model
    model = models.Sequential()
    model.add(layers.TimeDistributed(cnn, input_shape=(30, 160, 160, 3)))
    model.add(layers.LSTM(30, return_sequences=True))
    model.add(layers.TimeDistributed(layers.Dense(90)))
    model.add(layers.Dropout(0.1))
    model.add(layers.GlobalAveragePooling1D())
    model.add(layers.Dense(512, activation='relu'))
    model.add(layers.Dropout(0.3))
    model.add(layers.Dense(num_classes, activation="sigmoid"))

    # Compile the model
    adam = optimizers.Adam(learning_rate=0.0005, beta_1=0.9, beta_2=0.999, epsilon=1e-08)
    model.compile(loss='binary_crossentropy', optimizer=adam, metrics=["accuracy"])

    return model

# Example usage
model22 = mamon_videoFightModel2()

# Make sure the model is compiled (already done in the function)
model22.summary()

# Example input data for prediction
import numpy as np
example_input = np.random.random((1, 30, 160, 160, 3))  # Shape: (batch_size, time_steps, height, width, channels)

# Make predictions
predictions = model22.predict(example_input)
print(predictions)


Model: "sequential_22"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed_16             │ (None, 30, 12800)      │    20,024,384 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_8 (LSTM)                   │ (None, 30, 30)         │     1,539,720 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_17             │ (None, 30, 90)         │         2,790 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 30, 90)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_8      │ (None, 90)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 512)            │        46,592 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 2)              │         1,026 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,614,512 (82.45 MB)

 Trainable params: 1,590,128 (6.07 MB)

 Non-trainable params: 20,024,384 (76.39 MB)

1/1 ━━━━━━━━━━━━━━━━━━━━ 12s 12s/step
[[0.49442598 0.508641  ]]


In [14]:
from tensorflow.keras import optimizers

import numpy as np
from skimage.transform import resize
np.random.seed(1234)
model22 = mamon_videoFightModel2(tf)


In [17]:
model22._make_predict_function()


AttributeError: 'Sequential' object has no attribute '_make_predict_function'

In [22]:
def video_mamonreader(cv2,filename):
    frames = np.zeros((30, 160, 160, 3), dtype=np.float64)
    i=0
    print(frames.shape)
    vc = cv2.VideoCapture(filename)
    if vc.isOpened():
        rval , frame = vc.read()
    else:
        rval = False
    frm = resize(frame,(160,160,3))
    frm = np.expand_dims(frm,axis=0)
    if(np.max(frm)>1):
        frm = frm/255.0
    frames[i][:] = frm
    i +=1
    print("reading video")
    while i < 30:
        rval, frame = vc.read()
        frm = resize(frame,(160,160,3))
        frm = np.expand_dims(frm,axis=0)
        if(np.max(frm)>1):
            frm = frm/255.0
        frames[i][:] = frm
        i +=1
    return frames

In [23]:
def pred_fight(model,video,acuracy=0.9):
    pred_test = model.predict(video)
    if pred_test[0][1] >=acuracy:
        return True , pred_test[0][1]
    else:
        return False , pred_test[0][1]

In [26]:
def main_fight(vidoss):
    vid = video_mamonreader(cv2,vidoss)
    datav = np.zeros((1, 30, 160, 160, 3), dtype=np.float64)
    datav[0][:][:] = vid
    millis = int(round(time.time() * 1000))
    print(millis)
    f , precent = pred_fight(model22,datav,acuracy=0.65)
    millis2 = int(round(time.time() * 1000))
    print(millis2)
    res_mamon = {'fight':f , 'precentegeoffight':str(precent)}
    res_mamon['processing_time'] =  str(millis2-millis)
    return res_mamon

In [27]:
res = main_fight('hdfight.mp4')

(30, 160, 160, 3)
reading video
1721894436628
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step
1721894442598


In [28]:
res

{'fight': False, 'precentegeoffight': '0.51767457', 'processing_time': '5970'}

In [29]:
res = main_fight('hdfight.mp4')

(30, 160, 160, 3)
reading video
1721894462027
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step
1721894467956


In [30]:
res

{'fight': False, 'precentegeoffight': '0.51767457', 'processing_time': '5929'}